# 03. Exploratory Review of Forecasting Results

Review saved backtest outputs without fitting models. Prepare processed data and run the current forecasting script before using this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the project results/ directory")
RESULTS = ROOT / "results"
files = {"backtest": RESULTS / "backtest_results.csv",
         "convergence": RESULTS / "sarima_convergence.csv",
         "fallbacks": RESULTS / "fallbacks.csv"}
missing = [str(path) for path in files.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing result files: " + ", ".join(missing) + ". Run src/forecasting/no_ml_predictions_v4.py first.")
backtest = pd.read_csv(files["backtest"], parse_dates=["origin", "date"])
convergence = pd.read_csv(files["convergence"], parse_dates=["origin"])
fallbacks = pd.read_csv(files["fallbacks"], parse_dates=["origin"])
print("Backtest shape:", backtest.shape)
print("Convergence shape:", convergence.shape)
print("Fallback shape:", fallbacks.shape)

## 1. Result Schemas and Coverage

In [ ]:
display(backtest.head())
display(backtest.dtypes.rename("Data type").to_frame())
print("Models:", sorted(backtest["model"].dropna().unique()))
print("Products (source names):", sorted(backtest["good"].dropna().unique()))
print("Forecast horizons:", sorted(backtest["h"].dropna().unique()) if "h" in backtest else "No h column")

## 2. Forecast Accuracy
Interpret each metric according to its definition in the forecasting script. Compare absolute errors across models within the same product.

In [ ]:
metric_cols = [c for c in ["ae", "se", "ase", "smape"] if c in backtest.columns]
summary_keys = [c for c in ["good", "model"] if c in backtest.columns]
if metric_cols:
    display(backtest.groupby(summary_keys)[metric_cols].mean().sort_values(metric_cols[0]))
else:
    display(backtest.groupby(summary_keys).size().to_frame("Count"))
if {"good", "model", "h"}.issubset(backtest.columns) and metric_cols:
    display(backtest.pivot_table(index=["good", "h"], columns="model", values=metric_cols[0], aggfunc="mean"))
if "regime" in backtest.columns and metric_cols:
    display(backtest.groupby("regime")[metric_cols].mean())

## 3. SARIMA Convergence and Fallbacks

In [ ]:
if "converged" in convergence:
    display(convergence.groupby("good")["converged"].agg(["count", "mean", "sum"]).rename(columns={"mean": "Convergence share", "sum": "Successful folds"}))
    print("Overall convergence share:", convergence["converged"].mean())
display(fallbacks.head(20))

## Chart Space

### Actual Values and Forecasts
Add actual-versus-forecast charts for selected products and rolling origins, with dates and units.

### Error by Horizon and Model
Add MAE/MASE/sMAPE plots by forecast step and a model ranking table.

### Convergence and Fallbacks
Visualize SARIMA convergence by product/origin and summarize fallback reasons.

## Interpretation, Limitations, and Findings
- Best model for each product: to be completed.
- Stability across horizons and regimes: to be completed.
- Backtest limitations and next steps: to be completed.

Note: original product names are preserved as supplied in the source data.